# 24 · Load: job and major code references (SOC, OEWS hybrids, CIP, CIP↔SOC) → `staging`

**Source card:** `crosswalks.md` (§6 issues, §7 staging tables). Explored in: notebook 06.

Reference tables, loaded as published (one table per file). The rules that use them (D2 degree-level filter, split weights for SOC 2010 → 2018) live in core.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_bls__soc_2018_structure` | one SOC 2018 code, any level | 1,447 |
| `stg_bls__soc_2010_2018` | one SOC 2010 → SOC 2018 code pair | 900 |
| `stg_bls__oews_hybrid` | one OEWS 2019 code × its SOC 2018 / OEWS 2018 / SOC 2010 parts | 868 |
| `stg_nces__cip_2020` | one CIP 2020 list code (2-, 4- or 6-digit) | 2,848 |
| `stg_nces__cip_2010_2020` | one detailed CIP 2010 code → its CIP 2020 code | 1,720 |
| `stg_nces__cip_soc` | one row of the official CIP-SOC sheet (links + "no match" rows) | 6,097 |

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "24-load-crosswalks.ipynb"
SOC, CIP = r"\d\d-\d{4}", r"\d\d\.\d{4}"
conn = connect()
summary = []
F = {k: raw_files(k) for k in ["bls_soc_structure", "bls_soc_crosswalk", "bls_oews_hybrid", "nces_cip"]}
path = lambda key, name: F[key][F[key].file.str.endswith(name)].path.iloc[-1]
fname = lambda key, name: F[key][F[key].file.str.endswith(name)].file.iloc[-1]

def strip_marks(s: pd.Series) -> pd.Series:
    return s.str.replace(r"\s*\(#+\)", "", regex=True).str.strip()

## 1. SOC 2018 structure (issue 1)

Header on row 8. Each code sits in the column of its level, and the file lists every code right after its parents, so parent codes are carried forward.

In [2]:
st = pd.read_excel(path("bls_soc_structure", "soc_structure_2018.xlsx"), header=7, dtype=str, keep_default_na=False)
st = st.apply(lambda s: s.str.strip())
st.columns = ["major", "minor", "broad", "detailed", "title"]
is_code = st[["major", "minor", "broad", "detailed"]].agg("".join, axis=1).str.fullmatch(SOC)
log("keep code rows (drop notes)", len(st), is_code.sum())
st = st[is_code].copy()
st["level"] = st[["major", "minor", "broad", "detailed"]].ne("").idxmax(axis=1)
st["soc_code"] = st[["major", "minor", "broad", "detailed"]].agg("".join, axis=1)
for lvl in ["major", "minor", "broad"]:
    st[f"{lvl}_code"] = st[lvl].replace("", pd.NA).ffill()
    st.loc[st.level == lvl, f"{lvl}_code"] = st.soc_code     # a code is its own parent at its level
st.loc[st.level == "major", ["minor_code", "broad_code"]] = pd.NA
st.loc[st.level == "minor", "broad_code"] = pd.NA
assert (st.soc_code.str[:2] == st.major_code.str[:2]).all()
print(st.level.value_counts().to_dict()); assert st.level.value_counts().to_dict() == {"detailed": 867, "broad": 459, "minor": 98, "major": 23}
soc = st.rename(columns={"title": "soc_title"})[["soc_code", "soc_title", "level", "major_code", "minor_code", "broad_code"]]
soc = soc.assign(source_file=fname("bls_soc_structure", "soc_structure_2018.xlsx")).reset_index(drop=True)
summary.append(stage(soc, "stg_bls__soc_2018_structure", ["soc_code"], expected_rows=1_447, source_key="bls_soc_structure",
                     notebook=NB, conn=conn, comment="SOC 2018 hierarchy: 23 major, 98 minor, 459 broad, 867 detailed codes."))

  keep code rows (drop notes): 1,447 -> 1,447 rows (+0)
{'detailed': 867, 'broad': 459, 'minor': 98, 'major': 23}
  stg_bls__soc_2018_structure: 1,447 rows x 7 cols, key ['soc_code'] unique | local data/interim/staging/stg_bls__soc_2018_structure.parquet | DB: 1447


## 2. SOC 2010 → 2018 (issues 4–5)

Header on row 9. `(#)` / `(##)` marks are stripped from titles; the `relation` column (1:1, split, merge, split + merge) carries what they meant. Split weights are decided in core.

In [3]:
x = pd.read_excel(path("bls_soc_crosswalk", "soc_2010_to_2018_crosswalk.xlsx"), header=8, dtype=str, keep_default_na=False)
x = x.apply(lambda s: s.str.strip())
x.columns = ["soc_2010", "soc_2010_title", "soc_2018", "soc_2018_title"]
before = len(x); x = x[x.soc_2010.str.fullmatch(SOC) & x.soc_2018.str.fullmatch(SOC)]
log("keep code pairs", before, len(x))
print("Titles with a (#) mark:", int(x.soc_2010_title.str.contains(r"\(#").sum() + x.soc_2018_title.str.contains(r"\(#").sum()))
for c in ["soc_2010_title", "soc_2018_title"]:
    x[c] = strip_marks(x[c])
n18 = x.groupby("soc_2010").soc_2018.transform("nunique")
n10 = x.groupby("soc_2018").soc_2010.transform("nunique")
x["relation"] = (n18.gt(1).astype(int) + 2 * n10.gt(1).astype(int)).map({0: "1:1", 1: "split", 2: "merge", 3: "split + merge"})
print(x.relation.value_counts().to_dict())
assert x.relation.value_counts().to_dict() == {"1:1": 766, "split": 70, "split + merge": 32, "merge": 32}
assert set(x.soc_2018) == set(soc[soc.level == "detailed"].soc_code)
x["source_file"] = fname("bls_soc_crosswalk", "soc_2010_to_2018_crosswalk.xlsx")
summary.append(stage(x.reset_index(drop=True), "stg_bls__soc_2010_2018", ["soc_2010", "soc_2018"], expected_rows=900,
                     source_key="bls_soc_crosswalk", notebook=NB, conn=conn,
                     comment="BLS SOC 2010 -> 2018 code pairs with relation (1:1 / split / merge / split + merge)."))

  keep code pairs: 900 -> 900 rows (+0)
Titles with a (#) mark: 166
{'1:1': 766, 'split': 70, 'split + merge': 32, 'merge': 32}


  stg_bls__soc_2010_2018: 900 rows x 6 cols, key ['soc_2010', 'soc_2018'] unique | local data/interim/staging/stg_bls__soc_2010_2018.parquet | DB: 900


## 3. OEWS 2019 hybrid structure (issues 3, 6)

Header on row 6. 889 padded title/note cells, stripped. Rows are kept as published; blank parts stay NULL. 30 OEWS codes are neither SOC 2018 nor SOC 2010 (BLS aggregates); 5 are still in current files.

In [4]:
h = pd.read_excel(path("bls_oews_hybrid", "oes_2019_hybrid_structure.xlsx"), header=5, dtype=str, keep_default_na=False)
padded = int((h != h.apply(lambda s: s.str.strip())).sum().sum())
h = h.apply(lambda s: s.str.strip())
print(f"Padded cells stripped: {padded}")
h.columns = ["oes_2019", "oes_2019_title", "soc_2018", "soc_2018_title", "oes_2018", "oes_2018_title", "soc_2010",
             "soc_2010_title", "notes"]
before = len(h); h = h[h.oes_2019.str.fullmatch(SOC)]
log("keep rows with an OEWS 2019 code", before, len(h))
blanks = {c: int((h[c] == "").sum()) for c in ["soc_2018", "oes_2018", "soc_2010"]}
print("Blank part codes (-> NULL):", blanks)
KEY_H = ["oes_2019", "soc_2018", "oes_2018", "soc_2010"]
assert not any(blanks.values()), "a part code is blank: the PK would need a sentinel"
h = h.replace({"": None})
h["is_bls_aggregate"] = ~h.oes_2019.isin(set(soc.soc_code)) & ~h.oes_2019.isin(set(x.soc_2010))
agg19 = set(h[h.is_bls_aggregate].oes_2019)
agg18 = set(h.oes_2018) - set(soc.soc_code) - set(x.soc_2010)
print(f"BLS aggregate codes (neither SOC 2018 nor SOC 2010): OEWS 2019 side {len(agg19)}, "
      f"both sides {len(agg19 | agg18)} (nb 06 counted both sides: 30)")
assert len(agg19 | agg18) == 30
# 4 of the 5 aggregates still in current files are here; the 5th, 25-2052, is a real SOC 2010 code BLS kept
# (so this test can't see it). The job dimension in core lists all 5 explicitly.
assert {"21-1018", "25-9045", "51-2028", "53-1047"} <= agg19 and "25-2052" in set(x.soc_2010)
h["source_file"] = fname("bls_oews_hybrid", "oes_2019_hybrid_structure.xlsx")
summary.append(stage(h.reset_index(drop=True), "stg_bls__oews_hybrid", KEY_H, expected_rows=868,
                     source_key="bls_oews_hybrid", notebook=NB, conn=conn,
                     comment="OEWS May 2019-2020 hybrid codes and their SOC 2018 / OEWS 2018 / SOC 2010 parts."))

Padded cells stripped: 889
  keep rows with an OEWS 2019 code: 868 -> 868 rows (+0)
Blank part codes (-> NULL): {'soc_2018': 0, 'oes_2018': 0, 'soc_2010': 0}
BLS aggregate codes (neither SOC 2018 nor SOC 2010): OEWS 2019 side 27, both sides 30 (nb 06 counted both sides: 30)
  stg_bls__oews_hybrid: 868 rows x 11 cols, key ['oes_2019', 'soc_2018', 'oes_2018', 'soc_2010'] unique | local data/interim/staging/stg_bls__oews_hybrid.parquet | DB: 868


## 4. CIP 2020 list (issues 2, 8)

Codes are wrapped in `="…"` so Excel keeps leading zeros; stripped and pattern-checked. The list stacks 2-digit families, 4-digit series and 6-digit codes, including old codes that moved or were deleted. In this file, `Moved to` marks the new code (the opposite of the crosswalk), so core must use code pairs from §5, never this label.

In [5]:
unwrap = lambda d: d.apply(lambda s: s.str.strip().str.removeprefix('="').str.removesuffix('"').str.strip())
c20 = unwrap(pd.read_csv(path("nces_cip", "CIPCode2020.csv"), dtype=str, keep_default_na=False))
c20["level"] = c20.CIPCode.map(lambda c: "detailed" if pd.Series([c]).str.fullmatch(CIP).iloc[0]
                               else "series" if pd.Series([c]).str.fullmatch(r"\d\d\.\d\d").iloc[0]
                               else "family" if pd.Series([c]).str.fullmatch(r"\d\d").iloc[0] else None)
assert c20.level.notna().all(), c20[c20.level.isna()].CIPCode.unique()
print(c20.level.value_counts().to_dict(), "| Action:", c20.Action.value_counts().to_dict())
cip20 = (c20.rename(columns={"CIPCode": "cip_code", "CIPFamily": "cip_family", "Action": "action", "TextChange": "text_change",
                             "CIPTitle": "cip_title", "CIPDefinition": "definition", "CrossReferences": "cross_references",
                             "Examples": "examples"})
         [["cip_code", "cip_family", "level", "cip_title", "action", "text_change", "definition", "cross_references", "examples"]]
         .replace({"": None}).assign(source_file=fname("nces_cip", "CIPCode2020.csv")))
summary.append(stage(cip20, "stg_nces__cip_2020", ["cip_code"], expected_rows=2_848, source_key="nces_cip",
                     notebook=NB, conn=conn, comment="NCES CIP 2020 list: families, series, detailed codes (incl. moved/deleted old codes)."))

{'detailed': 2325, 'series': 473, 'family': 50} | Action: {'No substantive changes': 1994, 'New': 544, 'Moved from': 149, 'Moved to': 149, 'Deleted': 12}
  stg_nces__cip_2020: 2,848 rows x 10 cols, key ['cip_code'] unique | local data/interim/staging/stg_nces__cip_2020.parquet | DB: 2848


## 5. CIP 2010 → 2020 (issues 2, 8, 9)

Detailed codes only, one row per CIP 2010 code (strictly 1:1, asserted). `Deleted` rows repeat the old code as the 2020 code: the target is set to NULL. The 460 `New` codes have no 2010 code; they're in `stg_nces__cip_2020`.

In [6]:
cx = unwrap(pd.read_csv(path("nces_cip", "Crosswalk2010to2020.csv"), dtype=str, keep_default_na=False))
det = cx[cx.CIPCode2010.str.fullmatch(CIP)].copy()
log("keep rows with a detailed CIP 2010 code", len(cx), len(det))
print(det.Action.value_counts().to_dict())
deleted = det.Action == "Deleted"
det.loc[deleted, ["CIPCode2020", "CIPTitle2020"]] = None
print(f"Deleted: 2020 target set to NULL on {deleted.sum()} rows")
assert det.loc[~deleted, "CIPCode2020"].str.fullmatch(CIP).all()
assert not det.loc[~deleted].CIPCode2020.duplicated().any(), "a 2020 code receives two 2010 codes"
cip_x = (det.rename(columns={"CIPCode2010": "cip_2010", "CIPTitle2010": "cip_2010_title", "Action": "action",
                             "Text change": "text_change", "CIPCode2020": "cip_2020", "CIPTitle2020": "cip_2020_title"})
         .replace({"": None}).assign(source_file=fname("nces_cip", "Crosswalk2010to2020.csv")).reset_index(drop=True))
summary.append(stage(cip_x, "stg_nces__cip_2010_2020", ["cip_2010"], expected_rows=1_720, source_key="nces_cip",
                     notebook=NB, conn=conn, comment="NCES CIP 2010 -> 2020, detailed codes, strictly 1:1; NULL target = deleted."))

  keep rows with a detailed CIP 2010 code: 2,699 -> 1,720 rows (-979)
{'No substantive changes': 1568, 'Moved to': 145, 'Deleted': 7}
Deleted: 2020 target set to NULL on 7 rows
  stg_nces__cip_2010_2020: 1,720 rows x 7 cols, key ['cip_2010'] unique | local data/interim/staging/stg_nces__cip_2010_2020.parquet | DB: 1720


## 6. CIP 2020 ↔ SOC 2018 (issues 10–11)

The `CIP-SOC` sheet is the master (the `SOC-CIP` sheet misses 4 links for `03.0204`). It holds real links plus "no match" rows: `99-9999` = a major with no job, `99.9999` = a job with no major. Kept with a `link_type` column so core can turn them into flags on the dimensions.

In [7]:
xl = path("nces_cip", "CIP2020_SOC2018_Crosswalk.xlsx")
cs = pd.read_excel(xl, sheet_name="CIP-SOC", dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
cs.columns = ["cip_code", "cip_title", "soc_code", "soc_title"]
assert cs.cip_code.str.fullmatch(CIP).all() and cs.soc_code.str.fullmatch(SOC).all()
cs["link_type"] = "link"
cs.loc[cs.soc_code == "99-9999", "link_type"] = "major_no_job"
cs.loc[cs.cip_code == "99.9999", "link_type"] = "job_no_major"
print(cs.link_type.value_counts().to_dict())
assert (cs.link_type == "link").sum() == 5_723

sc = pd.read_excel(xl, sheet_name="SOC-CIP", dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
pairs = lambda d: set(zip(d.CIP2020Code, d.SOC2018Code))
only_cs = {(c, s) for c, s in zip(cs.cip_code, cs.soc_code)} - pairs(sc)
print("Links in CIP-SOC but not in SOC-CIP:", sorted(only_cs))
cs["source_file"] = fname("nces_cip", "CIP2020_SOC2018_Crosswalk.xlsx") + "!CIP-SOC"
summary.append(stage(cs, "stg_nces__cip_soc", ["cip_code", "soc_code"], expected_rows=6_097, source_key="nces_cip",
                     notebook=NB, conn=conn, comment="NCES CIP 2020 <-> SOC 2018 crosswalk (CIP-SOC sheet): 5,723 links + no-match rows (link_type)."))

{'link': 5723, 'major_no_job': 194, 'job_no_major': 180}


Links in CIP-SOC but not in SOC-CIP: [('03.0204', '11-9199'), ('03.0204', '19-3011'), ('03.0204', '19-4061'), ('03.0204', '25-1063')]


  stg_nces__cip_soc: 6,097 rows x 6 cols, key ['cip_code', 'soc_code'] unique | local data/interim/staging/stg_nces__cip_soc.parquet | DB: 6097


## 7. Summary

In [8]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__soc_2018_structure,1447,7,soc_code,bls_soc_structure,24-load-crosswalks.ipynb,1447,2026-10-14T21:12:27+00:00
1,stg_bls__soc_2010_2018,900,6,soc_2010 + soc_2018,bls_soc_crosswalk,24-load-crosswalks.ipynb,900,2026-10-14T21:12:27+00:00
2,stg_bls__oews_hybrid,868,11,oes_2019 + soc_2018 + oes_2018 + soc_2010,bls_oews_hybrid,24-load-crosswalks.ipynb,868,2026-10-14T21:12:27+00:00
3,stg_nces__cip_2020,2848,10,cip_code,nces_cip,24-load-crosswalks.ipynb,2848,2026-10-14T21:12:28+00:00
4,stg_nces__cip_2010_2020,1720,7,cip_2010,nces_cip,24-load-crosswalks.ipynb,1720,2026-10-14T21:12:28+00:00
5,stg_nces__cip_soc,6097,6,cip_code + soc_code,nces_cip,24-load-crosswalks.ipynb,6097,2026-10-14T21:12:28+00:00
